# Nível 1 — Exclusões procedimentais

Pablo Rogers

In [ ]:
#| label: setup-01
#| include: false

library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

## Leitura e preparação dos dados

O arquivo `data.csv` contém as respostas brutas do questionário (separador `;`). Nesta etapa, removem-se registros de abandono, selecionam-se as variáveis do fluxo analítico e deriva-se o tempo total de resposta (`TIME`).

In [ ]:
#| label: leitura-dados

dados_brutos <- read_csv2(
  "../../Data/InputData/data.csv",
  show_col_types = FALSE
)

ℹ Using "','" as decimal and "'.'" as grouping mark. Use `read_delim()` for more control.

Rows: 1,326
Columns: 33
$ ID       <dbl> 10441016017, 10440981694, 10440601239, 10440562981, 104403585…
$ CREATED  <chr> "28/12/2018 12:00", "28/12/2018 11:25", "28/12/2018 03:23", "…
$ MODIFIED <chr> "28/12/2018 12:20", "28/12/2018 11:47", "28/12/2018 03:39", "…
$ CQ1      <dbl> 1, 1, 1, 1, 1, 1, 1, 1, NA, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, …
$ CQ2      <dbl> 1, 1, 1, 1, 1, 1, 1, 1, NA, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, …
$ CQ3      <dbl> 1, 1, 1, 1, 1, 1, 1, 1, NA, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, …
$ Q1       <dbl> 5, 3, 1, 4, 3, 4, 3, 4, 2, 5, 4, 3, 4, 2, 4, 5, 4, 5, 4, 4, 4…
$ Q2       <dbl> 5, 4, 1, 4, 4, 4, 4, 4, 2, 5, 4, NA, 4, 1, 4, 5, 4, 4, 4, 4, …
$ Q3       <dbl> 1, 2, 3, 2, 1, 1, 1, 1, 4, 2, 3, 2, 2, 3, 3, 1, 2, 2, 2, 3, 1…
$ Q4       <dbl> 1, 2, 2, 1, 1, 1, 2, 1, 4, 1, 3, 2, 2, 2, 2, 2, 2, 1, 2, 1, 1…
$ Q5       <dbl> 3, 3, 1, 3, 3, 3, 3, 3, 1, 4, 3, 4, 4, 1, 3, 4, 3, 5, 3, 4, 3…
$ Q6       <dbl> 4, 4, 1, 3, 5, 4, 3, 4, 4, 4, 5, 3, 3, 4, 4, 5, 3, 5, 4, 4, 3…
$ Q7       <dbl>

## Exclusões procedimentais (Nível 1)

-   `CQ == 1`: o respondente acertou o item de controle de qualidade.
-   `CQ == 0`: o respondente falhou no item de controle de qualidade.

In [ ]:
#| label: nivel-1

dados_nivel1 <- dados_importados |>
  mutate(
    cq_falhas = rowSums(across(CQ1:CQ3, \(x) x != 1), na.rm = TRUE),
    na_count  = rowSums(across(Q1:Q26, is.na)),
    excl_time = TIME < 208,
    excl_cq   = cq_falhas >= 2,
    excl_na   = na_count > 5,
    nivel1_excluido = excl_time | excl_cq | excl_na
  )

exclusion_log <- bind_rows(
  dados_preparados |>
    filter(abandono) |>
    transmute(
      ID,
      criterio = "abandono",
      valor    = NA_real_,
      detalhe  = "Sem respostas válidas em Q1-Q26"
    ),
  dados_nivel1 |>
    filter(excl_time) |>
    transmute(
      ID,
      criterio = "time_rapido",
      valor    = TIME,
      detalhe  = paste0("TIME = ", TIME, " s")
    ),
  dados_nivel1 |>
    filter(excl_cq) |>
    transmute(
      ID,
      criterio = "cq_falhas",
      valor    = cq_falhas,
      detalhe  = paste0("Falhas CQ = ", cq_falhas)
    ),
  dados_nivel1 |>
    filter(excl_na) |>
    transmute(
      ID,
      criterio = "missings_excessivos",
      valor    = as.numeric(na_count),
      detalhe  = paste0("Missings em Q1-Q26 = ", na_count)
    )
) |>
  distinct(ID, criterio, .keep_all = TRUE)

dir.create("../../Output/Results", recursive = TRUE, showWarnings = FALSE)
write_csv2(exclusion_log, "../../Output/Results/exclusion_log.csv")

dados_elegiveis <- dados_nivel1 |>
  filter(!nivel1_excluido) |>
  select(ID, CQ1, CQ2, CQ3, TIME, Q1:Q26) |>
  # Inversão dos itens reversos (6 - valor)
  mutate(across(c(Q3, Q4, Q26), ~ 6 - .x))

### Sumário do Nível 1

In [ ]:
#| label: sumario-nivel-1

tibble(
  Etapa = c(
    "Dados brutos",
    "Abandono removido",
    "Excluídos por tempo (< 208 s)",
    "Excluídos por CQ (≥ 2 falhas)",
    "Excluídos por missings (> 5 em Q1-Q26)",
    "Elegíveis para Nível 2",
    "NAs restantes na escala Q1-Q26"
  ),
  N = c(
    nrow(dados_brutos),
    sum(dados_preparados$abandono, na.rm = TRUE),
    sum(dados_nivel1$excl_time, na.rm = TRUE),
    sum(dados_nivel1$excl_cq, na.rm = TRUE),
    sum(dados_nivel1$excl_na, na.rm = TRUE),
    nrow(dados_elegiveis),
    sum(is.na(dados_elegiveis |> select(Q1:Q26)))
  )
) |>
  knitr::kable(caption = "Resumo das exclusões procedimentais — Nível 1")

  Etapa                                          N
  ----------------------------------------- ------
  Dados brutos                                1546
  Abandono removido                            220
  Excluídos por tempo (\< 208 s)                 0
  Excluídos por CQ (≥ 2 falhas)                 28
  Excluídos por missings (\> 5 em Q1-Q26)        3
  Elegíveis para Nível 2                      1295
  NAs restantes na escala Q1-Q26                90

  : Resumo das exclusões procedimentais --- Nível 1


In [ ]:
#| label: exportacao-nivel-1
#| include: false

dir.create("../../Data/IntermediateData", recursive = TRUE, showWarnings = FALSE)
write_csv2(dados_elegiveis, "../../Data/IntermediateData/whoqol_imported.csv")